In [ ]:
from pathlib import Path

import torch
import gc

from rtnls_inference import (
    HeatmapRegressionEnsemble,
    SegmentationEnsemble,
)

## Segmentation of preprocessed images (ONNX)

Runs ONNX releases from `releases_new` on preprocessed RGB images in `samples/fundus/rgb`:
- artery-vein segmentation
- vessel segmentation
- optic disc segmentation
- fovea localisation

In [ ]:
ds_path = Path("../samples/fundus")
releases_path = Path("/mnt/oogergo/eyened/models/rtnls_vascx/releases_new")

rgb_path = ds_path / "rgb"
av_path = ds_path / "av"
vessels_path = ds_path / "vessels"
discs_path = ds_path / "discs"
overlays_path = ds_path / "overlays"

PROVIDERS = ["CUDAExecutionProvider", "CPUExecutionProvider"]

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
rgb_paths = sorted(rgb_path.glob("*.png"))
rgb_paths

In [ ]:
data = [{"id": p.stem, "image": str(p)} for p in rgb_paths]

In [ ]:
data

### Artery-vein segmentation

In [ ]:
av_ensemble = SegmentationEnsemble.from_onnx(
    releases_path / "av_wsoft_patches_02_finetune.onnx",
    providers=PROVIDERS
).to(device)

In [ ]:
print(av_ensemble.get_device())
print(av_ensemble.ensemble._session.get_providers())

In [ ]:
av_ensemble.predict_preprocessed(data, dest_path=av_path, num_workers=2)

In [ ]:
del av_ensemble
gc.collect()
torch.cuda.empty_cache()

### Vessel segmentation

In [ ]:
vessels_ensemble = SegmentationEnsemble.from_onnx(
    releases_path / "vessels_may26.onnx",
    providers=PROVIDERS
).to(device)
vessels_ensemble.predict_preprocessed(data, dest_path=vessels_path, num_workers=2)

In [ ]:
del vessels_ensemble
gc.collect()
torch.cuda.empty_cache()

### Disc segmentation

In [ ]:
disc_ensemble = SegmentationEnsemble.from_onnx(
    releases_path / "disc_may26.onnx",
    providers=PROVIDERS
).to(device)
disc_ensemble.predict_preprocessed(data, dest_path=discs_path, num_workers=2)

In [ ]:
del disc_ensemble
gc.collect()
torch.cuda.empty_cache()

### Fovea detection

In [ ]:
fovea_ensemble = HeatmapRegressionEnsemble.from_onnx(
    releases_path / "fovea_may26.onnx",
    providers=PROVIDERS
).to(device)
df = fovea_ensemble.predict_preprocessed(data, num_workers=2)
df.columns = ["mean_x", "mean_y"]
df.to_csv(ds_path / "fovea.csv")

In [ ]:
del fovea_ensemble
gc.collect()
torch.cuda.empty_cache()

### Plotting the retinas

Requires all models above to have been run and outputs stored under the folder names above.

In [ ]:
from vascx.fundus.loader import RetinaLoader

from rtnls_enface.utils.plotting import plot_gridfns

loader = RetinaLoader.from_folder(ds_path)

In [ ]:
loader[0].plot(av=True, disc=True, fovea=True, bounds=True)

In [ ]:
plot_gridfns([lambda ax: ret.plot(av=True, disc=True, fovea=True, bounds=True, ax=ax) for ret in loader])

### Storing visualizations (optional)

In [ ]:
from matplotlib import pyplot as plt

overlays_path.mkdir(parents=True, exist_ok=True)
for ret in loader:
    fig, ax = plt.subplots(1, 1, figsize=(8, 8), dpi=150)
    ret.plot(av=True, disc=True, fovea=True, bounds=True, ax=ax)
    fig.savefig(overlays_path / f"{ret.id}.png", bbox_inches="tight", pad_inches=0)
    plt.close(fig)